In [2]:
import pandas as pd
import numpy as np



In [3]:
orb = pd.read_json("https://minorplanetcenter.net/Extended_Files/mpcorb_extended.json.gz",compression='gzip')

In [4]:
# add column to DF which is the discovery year which is actually just the first four characters of Principal_desig
orb['discovery_linkage_year'] = pd.to_numeric(orb['Principal_desig'].str[0:4], errors='coerce')
# any discovery years greater than 2030 are bogus and should be NaN
orb.loc[orb['discovery_linkage_year'] > 2030, 'discovery_linkage_year'] = np.nan
orb['H_floor'] = orb['H'].apply(np.floor)

In [6]:
mbc_low_ecc = orb[(orb['e']<0.2) & (orb['a'].between(1.9,3.2))]
# inner is <2.5, middle is 2.5-2.8, outer is 2.8-3.2
def belt_class(row):
    if row['a'] < 2.5:
        return 'inner'
    elif 2.5 <= row['a'] < 2.8:
        return 'middle'
    else:
        return 'outer'
mbc_low_ecc['whichbelt'] = mbc_low_ecc.apply(belt_class, axis=1)

/var/folders/q_/q72gghqx0dbbrygb127lyk2m0000gp/T/ipykernel_2642/3104963315.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mbc_low_ecc['whichbelt'] = mbc_low_ecc.apply(belt_class, axis=1)


In [7]:
# group by whichbelt and H_floor, and give the average discovery year and average num_opps
mbc_grouped = mbc_low_ecc.groupby(['whichbelt', 'H_floor']).agg(
    avg_discovery_year=('discovery_linkage_year', 'mean'),
    avg_num_opps=('Num_opps', 'mean'),
    count_objects=('Principal_desig', 'count'),
    # Percent of objects with Num_opps = 1
    percent_num_opps_1=('Num_opps', lambda x: (x == 1).mean() * 100)
).reset_index()
mbc_grouped

,whichbelt,H_floor,avg_discovery_year,avg_num_opps,count_objects,percent_num_opps_1
0,inner,3.0,NaN,112.000000,1,0.000000
1,inner,6.0,NaN,91.666667,6,0.000000
2,inner,7.0,NaN,84.000000,11,0.000000
3,inner,8.0,NaN,72.250000,12,0.000000
4,inner,9.0,NaN,61.346154,26,0.000000
5,inner,10.0,1928.000000,50.656250,32,0.000000
6,inner,11.0,1938.772727,43.388235,85,0.000000
7,inner,12.0,1961.428135,34.718085,376,0.000000
8,inner,13.0,1981.038728,27.131653,1785,0.056022
9,inner,14.0,1990.872903,22.747938,6546,0.000000


In [10]:
output = mbc_grouped[(mbc_grouped['whichbelt']=='inner') & (mbc_grouped['H_floor'].between(16,21) )]

In [11]:
# Produce LaTeX table from the filtered inner-belt summary in `output`
inner_output = output.sort_values('H_floor').copy()


def format_year(value):
    return f"{int(round(value))}"


def format_oppositions(value):
    if value >= 10:
        return f"{value:.1f}"
    return f"{value:.2f}"


def format_percent(value):
    if value < 1:
        return f"{value:.2f}"
    if value < 10:
        return f"{value:.1f}"
    return f"{value:.0f}"


rows = []
for _, row in inner_output.iterrows():
    h_floor = int(row['H_floor'])
    hv_bin = f"{h_floor}--{h_floor + 1}"
    rows.append(
        f"{hv_bin} & {int(row['count_objects'])} & {format_year(row['avg_discovery_year'])} "
        f"& {format_oppositions(row['avg_num_opps'])} & {format_percent(row['percent_num_opps_1'])}\\% \\\\" 
    )

latex_lines = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Discovery and Observational Statistics by Absolute Magnitude for Known Inner Main-Belt Asteroids ($1.9 < a < 2.5$ AU, $e < 0.2$)}",
    r"\label{tab:inner_belt_stats}",
    r"\begin{tabular}{ccccc}",
    r"\hline \hline",
    r"$H_V$ & $n$ & \shortstack{Avg. Year \\ Discovered} & \shortstack{Avg. Observed \\ Oppositions} &",
    r"\shortstack{Pct. Single-\\Opposition} \\",
    r"\hline",
    *rows,
    r"\hline",
    r"\end{tabular}",
    r"\end{table}",
]

latex_table = "\n".join(latex_lines)
print(latex_table)

\begin{table}[ht]
\centering
\caption{Discovery and Observational Statistics by Absolute Magnitude for Known Inner Main-Belt Asteroids ($1.9 < a < 2.5$ AU, $e < 0.2$)}
\label{tab:inner_belt_stats}
\begin{tabular}{ccccc}
\hline \hline
$H_V$ & $n$ & \shortstack{Avg. Year \\ Discovered} & \shortstack{Avg. Observed \\ Oppositions} &
\shortstack{Pct. Single-\\Opposition} \\
\hline
16--17 & 37673 & 2000 & 17.3 & 0.16\% \\
17--18 & 66990 & 2007 & 13.5 & 0.19\% \\
18--19 & 98900 & 2012 & 8.13 & 1.4\% \\
19--20 & 69901 & 2017 & 3.63 & 23\% \\
20--21 & 11333 & 2020 & 1.31 & 83\% \\
21--22 & 1549 & 2018 & 1.01 & 99\% \\
\hline
\end{tabular}
\end{table}
